# 02 - Data preprocessing
---
**Objective:** Prepare the dataset for model training.


**Contents:**
- Perform feature engineering (time, amount, age, geographic, aggregation features)  
- Apply target encoding for categorical variables  
- Handle class imbalance using appropriate techniques  
- Save processed datasets for model training  

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import os
import sys
sys.path.append(os.path.abspath('../'))
from src.data.loader import load_csv, save_file
import src.features.engineering as fe

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print("Libraries imported successfully!")
print(f"Python version: {sys.version.split()[0]}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Python version: 3.13.5
Pandas version: 2.3.1
NumPy version: 2.3.2


In [2]:
# Load data
train = load_csv('../data/processed/train_cleaned.csv')
print()
test = load_csv('../data/processed/test_cleaned.csv')

👉 Loading: ../data/processed/train_cleaned.csv
DATASET INFORMATION
Shape: 1,296,675 rows × 19 columns
Memory usage: 788.67 MB

Column types:
object     9
int64      5
float64    5
Name: count, dtype: int64

Missing values: 0

👉 Loading: ../data/processed/test_cleaned.csv
DATASET INFORMATION
Shape: 555,719 rows × 19 columns
Memory usage: 338.01 MB

Column types:
object     9
int64      5
float64    5
Name: count, dtype: int64

Missing values: 0


## A. Feature Engineering

In [3]:
# Time
# hour, day_of_week, month, is_weekend, is_night
train = fe.create_time_features(train)
test = fe.create_time_features(test)

# Amount
# amt_log, amt_zscore
train = fe.create_amount_features(train)
test = fe.create_amount_features(test)

# Age
# age -> Drop dob after creating age
train = fe.create_age_feature(train)
test = fe.create_age_feature(test)

# Geo
# distance_km between merchant and user
train = fe.create_geo_features(train)
test = fe.create_geo_features(test)

# Aggregation
# merchant_freq, category_freq, city_freq
train = fe.create_aggregation_features(train)
test = fe.create_aggregation_features(test)

# Target Encoding
# ONLY APPLY ON TRAIN -> then map to test
# merchant_fraud_rate, category_fraud_rate, city_fraud_rate
for col in ['merchant', 'category', 'city']:
    train, test = fe.target_encode(train, test, col)

print("🚀 Feature engineering completed!")

🚀 Feature engineering completed!


## B. Imbalance data fixing

In [4]:
# Check class distribution
fraud_count = train['is_fraud'].sum()
total_count = len(train)
non_fraud_count = total_count - fraud_count

print("Class distribution:")
print(train['is_fraud'].value_counts())
print(f"Fraud ratio: {fraud_count / total_count:.6f}")

Class distribution:
is_fraud
0    1289169
1       7506
Name: count, dtype: int64
Fraud ratio: 0.005789


### Class Weight


In [5]:
# Compute scale_pos_weight for models like XGBoost
scale_pos_weight = non_fraud_count / fraud_count

print(f"scale_pos_weight: {scale_pos_weight:.2f}")

scale_pos_weight: 171.75


### SMOTE
- DO NOT save SMOTE data as final dataset
- Only use inside training pipeline

In [6]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder

# Drop unnecessary columns for modeling
X = train.drop(['is_fraud', 'trans_date_trans_time'], axis=1)
y = train['is_fraud']

# Encode categoricals
X_encoded = X.copy()

for col in X_encoded.select_dtypes(include='object').columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

# Apply SMOTE
smote = SMOTE(sampling_strategy=0.2, random_state=42)
X_res, y_res = smote.fit_resample(X_encoded, y)

print("After SMOTE:")
print(pd.Series(y_res).value_counts())

After SMOTE:
is_fraud
0    1289169
1     257833
Name: count, dtype: int64


### Save Processed Data

In [7]:
# Save processed datasets for next step (model training)
save_file(train, 'train_final.csv')
save_file(test, 'test_final.csv')

print("✅ Processed data saved successfully!")

File saved as: ../data/processed\train_final.csv
File saved as: ../data/processed\test_final.csv
✅ Processed data saved successfully!


## Conclusion

- Feature engineering was successfully applied, including time-based, transaction amount, age, geographic, and aggregation features.
- Target encoding was used to extract meaningful patterns from categorical variables such as merchant, category, and city.
- The dataset remains highly imbalanced, with fraudulent transactions representing a very small proportion of the data.
- Instead of modifying the dataset directly, imbalance handling will be addressed during model training using techniques such as class weighting and threshold tuning.
- The processed datasets have been saved and are ready for use in the model training phase.

**Next step:** Build and evaluate machine learning models using the processed data.